model diffusion

attempt1 Conditional Autoregressive Diffusion Model

Kaat: We gaan denk ik yield curves moeten genereren voor elke stimestep
Dus we moeten ook een dependency inbouwen rekening houdend met de yield curve van de vorige timestep


conditioneren op vorige yield curve zou sowieso moeten, maar hoe ver in de tijd? 


In [1]:
import os
import time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# Fix for common Jupyter/Windows crash
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ==========================================
# 1. DATA COLLECTION & PREPARATION (UPDATED)
# ==========================================
def get_dataset(filename='raw_data.csv'):
    df = pd.read_csv(filename, index_col=0)
    return df

def split_dataset(df, train_ratio=0.8):
    split_idx = int(len(df) * train_ratio)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

class SequentialYieldDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.macro_cols = ['FedFunds', 'CPI', 'GDP']
        self.yield_cols = [c for c in df.columns if 'Y_DGS' in c]

        # Normalisatie stats
        self.mu_y = df[self.yield_cols].values.mean()
        self.std_y = df[self.yield_cols].values.std()
        self.mu_c = df[self.macro_cols].mean()
        self.std_c = df[self.macro_cols].std()

        y_norm = (df[self.yield_cols].values - self.mu_y) / self.std_y
        c_norm = (df[self.macro_cols].values - self.mu_c.values) / self.std_c.values

        # Belangrijk: We maken paren (Y_t, [Macro_t, Y_{t-1}])
        # We verliezen de eerste rij omdat die geen 'vorige' dag heeft
        self.target_yields = torch.tensor(y_norm[1:], dtype=torch.float32)
        self.macro_conds = torch.tensor(c_norm[1:], dtype=torch.float32)
        self.prev_yields = torch.tensor(y_norm[:-1], dtype=torch.float32)

    def __len__(self):
        return len(self.target_yields)

    def __getitem__(self, idx):
        # Combineer Macro van nu met Yield van gisteren als één conditie vector
        condition = torch.cat([self.macro_conds[idx], self.prev_yields[idx]], dim=-1)
        return self.target_yields[idx], condition

# ==========================================
# 2. MODEL ARCHITECTURE (UPDATED cond_dim)
# ==========================================
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GroupNorm(8, dim),
            nn.SiLU(),
            nn.Linear(dim, dim)
        )
    def forward(self, x):
        return x + self.net(x)

class CondResNetSequential(nn.Module):
    def __init__(self, input_dim=11, cond_dim=14, hidden_dim=256): # cond_dim is nu 14
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.macro_mlp = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.res_blocks = nn.ModuleList([ResBlock(hidden_dim) for _ in range(4)])
        self.final_proj = nn.Linear(hidden_dim, input_dim)

    def forward(self, x, t, c):
        t_emb = self.time_mlp(t.float().unsqueeze(-1) / 1000.0)
        c_emb = self.macro_mlp(c)

        h = self.input_proj(x) + t_emb + c_emb
        for block in self.res_blocks:
            h = block(h)
        return self.final_proj(h)

# ==========================================
# 3. DIFFUSION ENGINE (SAME LOGIC)
# ==========================================
class DiffusionEngine:
    def __init__(self, model, T=300, device='cpu'):
        self.model = model
        self.T = T
        self.device = device
        self.beta = torch.linspace(1e-4, 0.02, T, device=device)
        self.alpha_hat = torch.cumprod(1.0 - self.beta, dim=0)

    def train_loss(self, x0, c):
        x0, c = x0.to(self.device), c.to(self.device)
        t = torch.randint(0, self.T, (x0.shape[0],), device=self.device)
        noise = torch.randn_like(x0)
        a_hat = self.alpha_hat[t].unsqueeze(-1)
        xt = torch.sqrt(a_hat) * x0 + torch.sqrt(1 - a_hat) * noise
        v_target = torch.sqrt(a_hat) * noise - torch.sqrt(1 - a_hat) * x0
        v_pred = self.model(xt, t, c)
        return F.mse_loss(v_pred, v_target)

    @torch.no_grad()
    def sample(self, c_scenario):
        c_scenario = c_scenario.to(self.device)
        n = c_scenario.shape[0]
        x = torch.randn((n, 11), device=self.device)
        for i in reversed(range(self.T)):
            t = torch.full((n,), i, dtype=torch.long, device=self.device)
            v = self.model(x, t, c_scenario)
            a_hat = self.alpha_hat[i]
            x0_recons = torch.sqrt(a_hat) * x - torch.sqrt(1 - a_hat) * v
            if i > 0:
                a_hat_prev = self.alpha_hat[i - 1]
                direction_xt = torch.sqrt(1 - a_hat_prev) * ((x - torch.sqrt(a_hat) * x0_recons) / torch.sqrt(1 - a_hat))
                x = torch.sqrt(a_hat_prev) * x0_recons + direction_xt
            else: x = x0_recons
        return x

# ==========================================
# 5. PATH GENERATION (NEW!)
# ==========================================
def generate_synthetic_paths(model, diffuser, dataset, start_yield_norm, future_macros_norm, steps=12):
    """
    Genereert een sequentieel pad van yield curves (bijv. voor de komende 12 maanden)
    """
    model.eval()
    current_yield = start_yield_norm.clone().unsqueeze(0).to(diffuser.device) # (1, 11)
    
    path = [current_yield]
    
    for t in range(steps):
        # Conditie = [Macro_t, Yield_{t-1}]
        cond = torch.cat([future_macros_norm[t].unsqueeze(0).to(diffuser.device), current_yield], dim=-1)
        
        # Genereer de volgende stap Y_t
        next_yield = diffuser.sample(cond)
        path.append(next_yield)
        current_yield = next_yield
        
    return torch.cat(path, dim=0)

# ==========================================
# 7. MAIN
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Data laden
    raw_df = get_dataset('raw_data.csv')
    train_df, test_df = split_dataset(raw_df)
    
    train_dataset = SequentialYieldDataset(train_df)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Model met conditie dim 14 (3 macro + 11 yield)
    model = CondResNetSequential(cond_dim=14).to(device)
    diffuser = DiffusionEngine(model, T=300, device=device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    # Training (epochs verlaagd voor demo)
    print("Start training...")
    for epoch in range(50):
        model.train()
        for y, c in train_loader:
            loss = diffuser.train_loss(y, c)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    # Demonstratie: Genereer een pad van 12 maanden
    print("\nPad genereren...")
    # Pak de laatste bekende curve uit de training set als startpunt
    start_curve = train_dataset.target_yields[-1]
    # Pak 12 maanden aan macro data
    future_macro = train_dataset.macro_conds[:12] 
    
    generated_path_norm = generate_synthetic_paths(model, diffuser, train_dataset, start_curve, future_macro)
    generated_path = (generated_path_norm.cpu().numpy() * train_dataset.std_y) + train_dataset.mu_y

    # Visualisatie van de evolutie van de 10Y rente (index 8)
    plt.plot(generated_path[:, 8], label="Gegenereerd Pad (10Y Yield)", marker='o')
    plt.title("Evolutie van 10Y Yield over 12 stappen")
    plt.ylabel("Yield %")
    plt.xlabel("Tijdstap")
    plt.legend()
    plt.show()
# Create a dictionary containing everything needed
checkpoint = {
    'model_state_dict': model.state_dict(),
    'stats': {
        'mu_y': train_dataset.mu_y,
        'std_y': train_dataset.std_y,
        'mu_c': train_dataset.mu_c,
        'std_c': train_dataset.std_c,
    },
    'hyperparameters': {
        'input_dim': 11,
        'cond_dim': 14,
        'hidden_dim': 256,
        'T': diffuser.T
    }
}

# Save to a file
torch.save(checkpoint, 'yield_diffusion_model.pt')
print("Model and statistics saved to yield_diffusion_model.pt")

FileNotFoundError: [Errno 2] No such file or directory: 'raw_data.csv'

2. How the recipient Loads the Model
The person you share the file with must have the class definition (CondResNetSequential and ResBlock) in their code. Then they can run this:



import torch
# 1. Load the checkpoint file
# Use map_location=torch.device('cpu') to ensure it works even if they don't have a GPU
checkpoint = torch.load('yield_diffusion_model.pt', map_location=torch.device('cpu'))

# 2. Reconstruct the model architecture with saved hyperparameters
h = checkpoint['hyperparameters']
loaded_model = CondResNetSequential(
    input_dim=h['input_dim'], 
    cond_dim=h['cond_dim'], 
    hidden_dim=h['hidden_dim']
)

# 3. Load the weights
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.eval() # Set to evaluation mode

# 4. Retrieve stats for scaling
stats = checkpoint['stats']
mu_y = stats['mu_y']
std_y = stats['std_y']

print("Model loaded successfully!")




3. Best Practices for Sharing
The .pt file is NOT the code: Your friend still needs your CondResNetSequential class code. They cannot just "open" the .pt file to see how the model works; it only contains numbers (weights).
Inference Script: It is helpful to provide a small "Inference Script" along with the .pt file that shows them exactly how to take a raw macro value, normalize it using the saved stats, run it through the model, and de-normalize the result.
Device Compatibility: Always suggest they use map_location when loading. If you saved the model on a GPU (cuda) and they try to load it on a laptop (cpu) without that argument, it will crash.
Zip it: If you want to be very professional, share a .zip containing:
model.py (The class definitions).
yield_diffusion_model.pt (The weights and stats).
generate.py (A simple script to produce a yield curve).

ALM model trainen voor 1 situatie (1  balans en 1 yield curve). die yield curve verandert dan in de tijd.

zouden we alles moeten 

HJM in krablicher: op t=0 curve van vandaag. voor elke time step 